# MS MARCO v2 embeddings on Colab (e5-base-v2 + PCA 192, float16)

Embeds the 138,364,198 MS MARCO v2 passages in **48 shards** (~2.9M each) straight from the raw `.gz` files (no
58 GB `corpus.jsonl` needed), and copies every finished shard to Google Drive. The rows, ids and text are exactly
what `prepare_dataset.py` + `collection_preproc.py` produce on the server, so the server can use them directly.

**Runtime:** *Runtime → Change runtime type → L4 GPU* (cheapest per passage; A100 is ~2x faster for a bit more).

**Resumable:** if the session dies, just *Run all* again: it re-downloads (~5-10 min) and skips shards already on
Drive. **Faster:** open this notebook in 2-3 tabs/sessions with `SESSION = 0, 1, 2` and `NUM_SESSIONS = 3`; each
does every 3rd shard.

When all 48 are on Drive: copy the Drive folder to the server's `datasets/msmarco-v2/local_import/` and run
`bash datasets/import_local_embeds.sh msmarco-v2` there (it merges them into float32 and embeds the queries).

In [ ]:
# ---- settings
DRIVE_DIR    = "/content/drive/MyDrive/pillar/msmarco-v2"   # shards + PCA end up here (~55 GB)
NUM_SHARDS   = 48
SESSION, NUM_SESSIONS = 0, 1        # running in 3 tabs? use SESSION = 0 / 1 / 2 and NUM_SESSIONS = 3
ENCODE_BATCH = 256                  # L4 / A100 / T4 all fit 256
REPO   = "https://github.com/dkblackley/bins-go.git"   # private repo? add a GITHUB_TOKEN in Colab's Secrets (key icon)
MARCO  = "https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco_v2_passage.tar"
EXPECTED_PASSAGES = 138_364_198

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess, shutil, glob, json, time
os.makedirs(DRIVE_DIR, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

In [ ]:
# ---- code + libraries (Colab already has torch)
if not os.path.exists("/content/bins-go"):
    url = REPO
    try:
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN")
        url = REPO.replace("https://", f"https://{tok}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "--depth", "1", url, "/content/bins-go"], check=True)
!pip -q install sentence-transformers faiss-cpu

In [ ]:
# ---- MS MARCO v2 passages: streamed straight out of the tar (no 20 GB tar on disk), ~5-10 min
RAW = "/content/msmarco/msmarco_v2_passage"
if len(glob.glob(f"{RAW}/msmarco_passage_*.gz")) < 70 or not os.path.exists(f"{RAW}/msmarco_passage_69.gz"):
    os.makedirs("/content/msmarco", exist_ok=True)
    t = time.time()
    subprocess.run(f"curl -sSL --retry 5 {MARCO} | tar -x -C /content/msmarco", shell=True, check=True)
    print(f"downloaded in {(time.time() - t) / 60:.1f} min")
# passage counts per file: computed once (~3 min), kept on Drive for the next session
if os.path.exists(f"{DRIVE_DIR}/line_counts.json") and not os.path.exists(f"{RAW}/line_counts.json"):
    shutil.copy(f"{DRIVE_DIR}/line_counts.json", f"{RAW}/line_counts.json")
os.chdir("/content/bins-go")
import collection_preproc as cp
n = sum(cp.raw_line_counts(cp.raw_msmarco_v2_files(RAW)))
shutil.copy(f"{RAW}/line_counts.json", f"{DRIVE_DIR}/line_counts.json")
print(f"{n:,} passages")
assert n == EXPECTED_PASSAGES, "passage count differs from the server's corpus, don't use these"

In [ ]:
# ---- PCA (768 -> 192): trained once on the first 100k passages (same sample as the server would use), shared by
# every shard and every session through Drive
OUT = "/content/out"; os.makedirs(OUT, exist_ok=True)
PCA = f"{OUT}/pca_e5_768_to_192.faiss"
EMB = f"{OUT}/msmarco-v2_corpus_embed_f16.npy"
if os.path.exists(f"{DRIVE_DIR}/pca_e5_768_to_192.faiss"):
    shutil.copy(f"{DRIVE_DIR}/pca_e5_768_to_192.faiss", PCA)
else:
    subprocess.run(["python", "collection_preproc.py", "--documents", RAW, "--output", EMB, "--pca", PCA,
                    "--pca-only"], check=True)
    if not os.path.exists(f"{DRIVE_DIR}/pca_e5_768_to_192.faiss"):   # another session may have beaten us
        shutil.copy(PCA, f"{DRIVE_DIR}/pca_e5_768_to_192.faiss")
    else:
        shutil.copy(f"{DRIVE_DIR}/pca_e5_768_to_192.faiss", PCA)
print("PCA ready")

In [ ]:
# ---- the shards: embed on local disk, copy to Drive (temp name + rename), delete locally
def on_drive(i):
    s = f"{DRIVE_DIR}/msmarco-v2_corpus_embed_f16.npy.shard{i:03d}of{NUM_SHARDS:03d}.npy"
    return os.path.exists(s) and os.path.exists(s + ".ids")

mine = [i for i in range(NUM_SHARDS) if i % NUM_SESSIONS == SESSION]
for i in mine:
    if on_drive(i):
        print(f"shard {i}: already on Drive"); continue
    t = time.time()
    subprocess.run(["python", "collection_preproc.py", "--documents", RAW, "--output", EMB, "--pca", PCA,
                    "--load-pca", "--num-docs", str(EXPECTED_PASSAGES), "--num-shards", str(NUM_SHARDS),
                    "--shard-id", str(i), "--dtype", "float16", "--fp16", "--encode-batch", str(ENCODE_BATCH),
                    "--skip-if-done"], check=True)
    local = f"{EMB}.shard{i:03d}of{NUM_SHARDS:03d}.npy"
    dst = f"{DRIVE_DIR}/{os.path.basename(local)}"
    for src, d in ((local + ".ids", dst + ".ids"), (local, dst)):   # .npy last: it marks the shard as done
        shutil.copy(src, d + ".uploading"); os.replace(d + ".uploading", d)
    for f in glob.glob(local + "*"):
        os.remove(f)
    print(f"shard {i}: done in {(time.time() - t) / 60:.1f} min")

In [ ]:
# ---- status
done = [i for i in range(NUM_SHARDS) if on_drive(i)]
print(f"{len(done)}/{NUM_SHARDS} shards on Drive; missing: {[i for i in range(NUM_SHARDS) if i not in done]}")
if len(done) == NUM_SHARDS:
    print("All done. Copy", DRIVE_DIR, "-> <server>/datasets/msmarco-v2/local_import/ (the shard .npy + .ids files")
    print("and pca_e5_768_to_192.faiss), then on the server: bash datasets/import_local_embeds.sh msmarco-v2")